# Can we fit $\mathcal{H}$ Using DNNs at a Single Kinematic Point Using the KM15/BKM10 Formalism for the Cross-Section **using Experimental Data Only**?

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Native Libraries:

In [ ]:
import os
import sys
import glob
import datetime

### (1.2): Import 3rd-Party Libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots
import numpy as np
import tensorflow as tf
import corner

from scipy.stats import norm
from sklearn.model_selection import train_test_split
import gepard as g
from gepard.fits import th_KM15

from bkm10_lib.core import DifferentialCrossSection
from bkm10_lib.inputs import BKM10Inputs
from bkm10_lib.cff_inputs import CFFInputs

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: tensorflow version: {tf.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")
print(f"[INFO]: corner version: {corner.__version__}")

### (1.4): Customizing Plotting Settings:

In [ ]:
plt.style.use('science')

## (2): Data Formatting/Collection Settings:

In [ ]:
VERSION_NUMBER = 1
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

## (3): Data Loading and Analysis:

### (3.1): Loading in `gepard`'s Datasets:

**Remark:** `g.dset` is a dictionary of `gepard` `DataSet` classes.
```python
g.dset[73]
> DataSet with 39 points
```

**Remark:** Some of the `DataPoint`s in the `DataSet`s *do not have all the right kinematics!* See below for this:

In [ ]:
try:
    print(f"[INFO]: k = {g.dset[47][0].in1energy}")
    print(f"[INFO]: xb = {g.dset[47][0].xB}")
    print(f"[INFO]: t = {g.dset[47][0].t}")
    print(f"[INFO]: Q^2 = {g.dset[47][0].Q2}")
except AttributeError:
    print(f"[ERROR]: Missing crucial kinematic setting information!")

### (3.2): Determining which `gepard` `DataSet`s have all of the right Kinematic Variables:

[NOTE]: We are looking for $k$, $x_{\textrm{B}}$, $t$, and $Q^{2}$.

In [ ]:
required_attributes = ['xB', 't', 'Q2', 'in1energy', 'observable', 'val', 'err']
valid_datasets = []

for dataset_index, dataset in g.dset.items():

    # check the first datapoint in the DataSet for contents!
    first_gepard_datapoint = dataset[0] if len(dataset) > 0 else None
    
    if first_gepard_datapoint and all(hasattr(first_gepard_datapoint, kinematic_attribute) for kinematic_attribute in required_attributes):
        valid_datasets.append(dataset_index)

print(f"[INFO]: Valid dataset indices are:\n{sorted(valid_datasets)}")
print(f"[INFO]: Length of valid datasets = {len(valid_datasets)}")
print(f"[INFO]: Compare length of *all* dataset = {len(g.dset)}")
print(f"[INFO]: Total invalid (according to our criteria) datasets = {len(g.dset) - len(valid_datasets)}")

According to [gepard documentation](https://gepard.phy.hr/docs/observables.htmlhttps://gepard.phy.hr/docs/observables.html), here are the relevant observables for our analysis

1. `XGAMMA` = basically DVCS cross-section
2. `XUU` = beam-averaged cross-section
1. `AC` = beam charge asymmetry
1. `ALU` = beam spin asymmetry
2. `AUL` = longitudinally-polarized target asymmetry
3. `AUT` = transversely-polarized target asymmetry

We need to figure out which `DataSet`s have the observables we're looking for!

### (3.3): Paritioning Datasets into Dictionary of Desired Observables

In [ ]:
target_observables = {'XGAMMA', 'XUU', 'AC', 'ALU', 'AUL'}

# dictionary of string-to-list key-value pairs:
desired_observable_dictionary = {observable: [] for observable in target_observables}

for dataset_index in valid_datasets:
    if not hasattr(g.dset[dataset_index][0], "observable"):
        print(f"[WARN]: Dataset {g.dset[dataset_index]} had datapoint without observable property...")
        continue
    
    observable_name = g.dset[dataset_index][0].observable
    
    if observable_name in desired_observable_dictionary:
        desired_observable_dictionary[observable_name].append(dataset_index)

for name, indices in desired_observable_dictionary.items():
    print(f"{name}: {indices}")

[NOTE]: From the brief code snippet above, we see that the *majority* of `gepard` `DataSet`s involve `"XUU"` and `"ALU"`, which justifies a simultaneous fit in the observables $d^{4}\sigma^{UU}$ and $\textrm{BSA}\left( 0 \right)$.

In [ ]:
xb_data = [p.xB for p in g.dset[100]]
q_squared_data = [p.Q2 for p in g.dset[100]]
t_data = [p.t for p in g.dset[100]]
label_plot = g.dset[100].observable

In [ ]:
#############################################
# figure initialization and customization
#############################################
input_space_figure = plt.figure(figsize = (9, 7))
input_space_axis = input_space_figure.add_subplot(1, 1, 1, projection = "3d")

#############################################
# figure/axis augmentation details:
#############################################
axis_elevation = input_space_axis.elev # extract eleveation param
axis_azimuthal = input_space_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
input_space_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = input_space_axis.transAxes)
input_space_axis.text2D(
    0.01, 0.00, 
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = input_space_axis.transAxes)

#############################################
# axis plotting data:
#############################################
input_space_axis.scatter(
    xb_data, q_squared_data, t_data,
    color = "blue", alpha = 0.7)

#############################################
# axis labeling data:
#############################################
input_space_axis.set_title(rf"Experimental Kinematic Settings Space for {label_plot}", fontsize = 18)
input_space_axis.set_xlabel(r"$x_{\textrm{B}}$", fontsize = 14)
input_space_axis.set_ylabel(r"$Q^{2}$", fontsize = 14)
input_space_axis.set_zlabel(r"$-t$", fontsize = 14)

#############################################
# plot saving configuration:
#############################################
input_space_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/experimental_kinematic_space_for_{label_plot}_v{MAJOR_MINOR_NUMBER}.png")
input_space_figure.savefig(f"./local/version_{MAJOR_MINOR_NUMBER}/plots/experimental_kinematic_space_for_{label_plot}_v{MAJOR_MINOR_NUMBER}.eps")
plt.close(input_space_figure)